In [1]:
# --- Core data handling ---
import pandas as pd
import numpy as np
import time

# --- RDKit: molecule parsing and Morgan fingerprints ---
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem import DataStructs

# --- Modeling ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score

# --- Plotting, for later sanity checks ---
import matplotlib.pyplot as plt

**Inspection of the subset**

In [2]:
# Load the ORIGINAL cleaned dataset (Phase 1 output), which still has the TD-DFT column
df_orig = pd.read_csv("beard_uvvis_cleaned.csv")

# Wait -- check: does this file actually still have the TD-DFT column, or did we drop it during export?
print(df_orig.columns.tolist())

['SMI', 'canonical_smi', 'lambda_max_exp_nm', 'extinction', 'solvent']


**Reload the raw CSV with the corrected schema, this time keeping the TD-DFT columns**

In [3]:
col_names = [
    "SMI",
    "lambda1_sTDA_nm",
    "F1_sTDA",
    "lambda1_TDDFT_nm",
    "F1_TDDFT",
    "lambda_max_exp_nm",
    "extinction",
    "solvent"
]

df_raw = pd.read_csv("paper_allDB.csv", header=None, skiprows=1, names=col_names)

print(f"Total rows: {df_raw.shape[0]}")
print(f"Rows with TD-DFT value present: {df_raw['lambda1_TDDFT_nm'].notna().sum()}")

Total rows: 8488
Rows with TD-DFT value present: 185


**Applying the same cleaning pipeline from Phase 1 to this TD-DFT subset — canonicalize SMILES, and critically, checking how many of these 185 compounds survived (and were removed by) all our earlier filtering steps**

In [4]:
from rdkit import Chem

df_tddft = df_raw[df_raw["lambda1_TDDFT_nm"].notna()].copy()

df_tddft["mol"] = df_tddft["SMI"].apply(Chem.MolFromSmiles)
df_tddft["is_valid"] = df_tddft["mol"].notna()
print(f"Valid SMILES among TD-DFT subset: {df_tddft['is_valid'].sum()} / {df_tddft.shape[0]}")

df_tddft = df_tddft[df_tddft["is_valid"]].copy()
df_tddft["canonical_smi"] = df_tddft["mol"].apply(Chem.MolToSmiles)

# Check overlap with our final, fully-cleaned Phase 1 dataset
df_clean_final = pd.read_csv("beard_uvvis_cleaned.csv")
overlap = df_tddft["canonical_smi"].isin(df_clean_final["canonical_smi"])
print(f"TD-DFT compounds that survived Phase 1 cleaning: {overlap.sum()} / {df_tddft.shape[0]}")

Valid SMILES among TD-DFT subset: 185 / 185
TD-DFT compounds that survived Phase 1 cleaning: 173 / 185


**Cheching how many of these 173 compounds were in our GP's training set versus held out in the test set**

In [5]:
# Recall X_gp / y_gp were built from the full 6,878-compound Phase 2 dataset, split with random_state=42
# We need the canonical_smi for every row in that split to check overlap properly

df_features_full = pd.read_csv("beard_model_ready_features.csv")

# Re-derive the same train/test split, but keep canonical_smi alongside this time
from sklearn.model_selection import train_test_split

indices = df_features_full.index
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_smis = set(df_features_full.loc[train_idx, "canonical_smi"])
test_smis = set(df_features_full.loc[test_idx, "canonical_smi"])

tddft_smis = set(df_tddft[overlap]["canonical_smi"])

in_train = tddft_smis & train_smis
in_test = tddft_smis & test_smis
in_neither = tddft_smis - train_smis - test_smis

print(f"TD-DFT compounds in GP training set: {len(in_train)}")
print(f"TD-DFT compounds in GP test set: {len(in_test)}")
print(f"TD-DFT compounds in neither (excluded by fingerprint scoping, e.g. MW>1000): {len(in_neither)}")

TD-DFT compounds in GP training set: 137
TD-DFT compounds in GP test set: 36
TD-DFT compounds in neither (excluded by fingerprint scoping, e.g. MW>1000): 0


**evaluating existing GP on just the 36 truly held-out compounds, and compute TD-DFT's own error on the same 36**

In [7]:
# Get the 36 held-out TD-DFT compounds' data
df_eval36 = df_tddft[df_tddft["canonical_smi"].isin(in_test)].copy()
print(f"Evaluation subset: {df_eval36.shape[0]} compounds")

# TD-DFT's own predictive error against experimental lambda_max
tddft_rmse = np.sqrt(mean_squared_error(df_eval36["lambda_max_exp_nm"], df_eval36["lambda1_TDDFT_nm"]))
tddft_r2 = r2_score(df_eval36["lambda_max_exp_nm"], df_eval36["lambda1_TDDFT_nm"])
print(f"TD-DFT vs experimental (n={df_eval36.shape[0]}): RMSE = {tddft_rmse:.2f} nm, R² = {tddft_r2:.3f}")

Evaluation subset: 36 compounds
TD-DFT vs experimental (n=36): RMSE = 108.70 nm, R² = 0.001


**Check for systematic bias, not just scatter — this is a different diagnostic than RMSE/R², and it matters a lot for TD-DFT specifically**

In [8]:
residuals = df_eval36["lambda_max_exp_nm"] - df_eval36["lambda1_TDDFT_nm"]

print(f"Mean signed error (bias): {residuals.mean():.2f} nm")
print(f"Median signed error: {residuals.median():.2f} nm")
print(f"Std of residuals: {residuals.std():.2f} nm")

# Quick look at the actual paired values
print(df_eval36[["canonical_smi", "lambda_max_exp_nm", "lambda1_TDDFT_nm"]].head(10))

Mean signed error (bias): 57.21 nm
Median signed error: 45.63 nm
Std of residuals: 93.74 nm
                                          canonical_smi  lambda_max_exp_nm  \
234                       c1ccc(-c2cnc(-c3ccccc3)o2)cc1              493.0   
266                      COc1ccc2cc(-c3ccsc3)c(=O)oc2c1              372.0   
555   O=S(=O)(c1ccccc1)n1c(C#Cc2ccccc2)cc2c(-c3ccccc...              220.0   
1152                         NC(=S)NN=Cc1c[nH]c2ccccc12              264.0   
1396                      Cn1c(C=Cc2ccccc2)nc(C#N)c1C#N              313.0   
1402  CCC1(CC)c2cc(C=C(C#N)C#N)ccc2-c2ccc(N(c3ccccc3...              518.6   
1512  c1ccc(-c2cc(-c3ccccc3)cc(-c3nnc(-c4cc(-c5ccccc...              260.0   
1568                     C(=NNc1ncnc2nc[nH]c12)c1ccccc1              268.0   
1681  Cc1ccc(C=Cc2ccc(-c3nc(-c4ccccc4)c(-c4ccccc4)n3...              365.0   
1704                      Oc1ccccc1-c1sc(-c2ccccn2)nc1O              377.0   

      lambda1_TDDFT_nm  
234         297.974460  

**GP on the same 36 compounds**

In [9]:
X_eval36 = pd.concat([
    fp_df.loc[df_features_full["canonical_smi"].isin(df_eval36["canonical_smi"])].reset_index(drop=True),
    df.loc[df_features_full["canonical_smi"].isin(df_eval36["canonical_smi"]), ["LargestConjugatedSystemSize"]].reset_index(drop=True)
], axis=1)

# Careful: need to match ordering exactly to df_eval36's experimental values
eval36_smis_ordered = df_features_full[df_features_full["canonical_smi"].isin(df_eval36["canonical_smi"])]["canonical_smi"].reset_index(drop=True)

X_eval36_scaled = scaler_gp.transform(X_eval36)
y_pred_eval36, y_std_eval36 = gp_full_local.predict(X_eval36_scaled, return_std=True)

gp_rmse_36 = np.sqrt(mean_squared_error(df_eval36.set_index("canonical_smi").loc[eval36_smis_ordered, "lambda_max_exp_nm"], y_pred_eval36))
gp_r2_36 = r2_score(df_eval36.set_index("canonical_smi").loc[eval36_smis_ordered, "lambda_max_exp_nm"], y_pred_eval36)

print(f"GP vs experimental (n=36, held-out): RMSE = {gp_rmse_36:.2f} nm, R² = {gp_r2_36:.3f}")
print(f"TD-DFT vs experimental (n=36):        RMSE = 108.70 nm, R² = 0.001")

NameError: name 'fp_df' is not defined